# Kaggle Notebook: 6 Baseline Defense Methods Evaluation
This notebook is fully self-contained and runs on Kaggle (with GPU accelerated environment).
It trains a standard ResNet-18 model on CIFAR-10, defines the 6 baseline defenses, and evaluates them on:
1. Clean images
2. Dense PGD-10 images
3. Sparse PGD (k=0.1) images

Defenses Evaluated:
1. Median Smoothing Defense (3x3 kernel)
2. Bit Reduction Defense (3-bit quantization)
3. JPEG Compression Defense (Quality=75)
4. Random Noise Injection Defense (std=0.02)
5. Randomized Smoothing (Monte Carlo expectation-based consensus wrapper)
6. Feature Denoising Defense (PyTorch forward hook-based spatial filtering)



In [1]:
import os
import io
import time
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as tv_models
from PIL import Image
import numpy as np
import pandas as pd
from tqdm import tqdm

# Set seed for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")



Using device: cuda


In [2]:
# 1. Define CIFAR-adapted ResNet-18
def make_cifar_resnet18(num_classes=10):
    model = tv_models.resnet18(num_classes=num_classes)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model



In [3]:
# 2. Define Attack Algorithms
def pgd_attack(model, images, labels, eps=8/255, alpha=2/255, iters=10):
    images = images.clone().detach().to(images.device)
    labels = labels.to(images.device)
    loss_fn = nn.CrossEntropyLoss()
    
    adv_images = images + torch.empty_like(images).uniform_(-eps, eps)
    adv_images = torch.clamp(adv_images, min=0, max=1).detach()
    
    for _ in range(iters):
        adv_images.requires_grad = True
        outputs = model(adv_images)
        loss = loss_fn(outputs, labels)
        grad = torch.autograd.grad(loss, adv_images, retain_graph=False, create_graph=False)[0]
        
        adv_images = adv_images.detach() + alpha * grad.sign()
        delta = torch.clamp(adv_images - images, min=-eps, max=eps)
        adv_images = torch.clamp(images + delta, min=0, max=1).detach()
    return adv_images

def topk_pgd_attack(model, images, labels, eps=8/255, alpha=2/255, iters=10, k_ratio=0.1, dynamic=True):
    images = images.clone().detach().to(images.device)
    labels = labels.to(images.device)
    loss_fn = nn.CrossEntropyLoss()
    
    adv_images = images.clone().detach()
    mask = None
    
    for t in range(iters):
        adv_images.requires_grad = True
        outputs = model(adv_images)
        loss = loss_fn(outputs, labels)
        grad = torch.autograd.grad(loss, adv_images, retain_graph=False, create_graph=False)[0]
        
        with torch.no_grad():
            if dynamic or (t == 0):
                if k_ratio == 0:
                    mask = torch.zeros_like(grad)
                else:
                    score = grad.abs()
                    score_flatten = score.view(score.size(0), -1)
                    N = score_flatten.size(1)
                    
                    k_max = int(N * k_ratio)
                    if dynamic:
                        k_t = int(k_max * (1 - t / iters))
                    else:
                        k_t = k_max
                    if k_t < 1: k_t = 1
                    
                    topk_vals, _ = torch.topk(score_flatten, k_t, dim=1)
                    tau = topk_vals[:, -1].view(-1, 1, 1, 1)
                    mask = (score >= tau).float()
                
            update = alpha * grad.sign() * mask
            adv_images.data.copy_(images + torch.clamp(adv_images + update - images, min=-eps, max=eps))
            adv_images.data.clamp_(0, 1)
            
    return adv_images




In [4]:
# 3. Define the 6 Preprocessing and Certified Defenses

class MedianSmoothingDefense(nn.Module):
    def __init__(self, kernel_size=3):
        super(MedianSmoothingDefense, self).__init__()
        self.kernel_size = kernel_size

    def forward(self, x):
        b, c, h, w = x.size()
        padding = self.kernel_size // 2
        x_pad = F.pad(x, (padding, padding, padding, padding), mode='reflect')
        patches = x_pad.unfold(2, self.kernel_size, 1).unfold(3, self.kernel_size, 1)
        patches = patches.contiguous().view(b, c, h, w, -1)
        median_val, _ = patches.median(dim=-1)
        return median_val

class BitReductionsDefense(nn.Module):
    def __init__(self, bits=3):
        super(BitReductionsDefense, self).__init__()
        self.levels = 2 ** bits

    def forward(self, x):
        return torch.round(x * (self.levels - 1)) / (self.levels - 1)

class JPEGCompressionDefense(nn.Module):
    def __init__(self, quality=75):
        super(JPEGCompressionDefense, self).__init__()
        self.quality = quality

    def forward(self, x):
        device = x.device
        b, c, h, w = x.size()
        x_np = x.cpu().numpy()
        x_defended = []
        for i in range(b):
            img_np = np.clip(x_np[i].transpose(1, 2, 0) * 255.0, 0, 255).astype(np.uint8)
            img = Image.fromarray(img_np)
            buffer = io.BytesIO()
            img.save(buffer, format="JPEG", quality=self.quality)
            buffer.seek(0)
            img_dec = Image.open(buffer)
            dec_np = np.array(img_dec).astype(np.float32) / 255.0
            x_defended.append(dec_np.transpose(2, 0, 1))
        return torch.tensor(np.array(x_defended), device=device, dtype=x.dtype)

class RandomNoiseDefense(nn.Module):
    def __init__(self, std=0.02):
        super(RandomNoiseDefense, self).__init__()
        self.std = std

    def forward(self, x):
        noise = torch.randn_like(x) * self.std
        return torch.clamp(x + noise, 0.0, 1.0)

class RandomizedSmoothingModel(nn.Module):
    def __init__(self, model, sigma=0.12, N=50): # N=50 for speed on demo
        super(RandomizedSmoothingModel, self).__init__()
        self.model = model
        self.sigma = sigma
        self.N = N

    def forward(self, x):
        B, C, H, W = x.size()
        x_expanded = x.unsqueeze(1).repeat(1, self.N, 1, 1, 1).view(B * self.N, C, H, W)
        noise = torch.randn_like(x_expanded) * self.sigma
        x_noisy = x_expanded + noise
        
        chunk_size = 64
        total_samples = B * self.N
        logits_list = []
        for i in range(0, total_samples, chunk_size):
            chunk_x = x_noisy[i:i+chunk_size]
            logits_chunk = self.model(chunk_x)
            logits_list.append(logits_chunk)
            
        logits = torch.cat(logits_list, dim=0)
        logits = logits.view(B, self.N, -1).mean(dim=1)
        return logits

class FeatureDenoisingWrapper(nn.Module):
    def __init__(self, model, kernel_size=3):
        super(FeatureDenoisingWrapper, self).__init__()
        self.model = model
        self.kernel_size = kernel_size
        self.hooks = []
        self._register_hooks()

    def _register_hooks(self):
        def denoise_hook(module, inp, out):
            B, C, H, W = out.size()
            padding = self.kernel_size // 2
            out_pad = F.pad(out, (padding, padding, padding, padding), mode='reflect')
            patches = out_pad.unfold(2, self.kernel_size, 1).unfold(3, self.kernel_size, 1)
            patches = patches.contiguous().view(B, C, H, W, -1)
            median_val, _ = patches.median(dim=-1)
            return median_val

        for name, module in self.model.named_modules():
            # Hook the second block layer and third block layer of standard ResNet-18
            if name in ['layer2', 'layer3']:
                h = module.register_forward_hook(denoise_hook)
                self.hooks.append(h)

    def remove_hooks(self):
        for h in self.hooks:
            h.remove()
        self.hooks = []

    def forward(self, x):
        return self.model(x)



In [5]:
# 4. Load Data
import shutil
import tarfile

def prepare_cifar10():
    target_dir = './data'
    extracted_target = os.path.join(target_dir, 'cifar-10-batches-py')
    tar_target = os.path.join(target_dir, 'cifar-10-python.tar.gz')
    
    if os.path.exists(extracted_target):
        print("CIFAR-10 is already prepared.")
        return
    
    os.makedirs(target_dir, exist_ok=True)
    candidates = [
        '/kaggle/input/datasets/truongnhatquangk18dn/aadata/cifar-10-python',
        '/kaggle/input/aadata/cifar-10-python',
        '/kaggle/input/aadata',
        '/kaggle/input/cifar-10-python'
    ]
    
    found = False
    for candidate in candidates:
        if not os.path.exists(candidate):
            continue
        
        # Check if contains extracted files directly
        if os.path.exists(os.path.join(candidate, 'batches.meta')):
            print(f"Linking extracted CIFAR-10 from {candidate}")
            os.symlink(candidate, extracted_target)
            found = True
            break
        elif os.path.exists(os.path.join(candidate, 'cifar-10-batches-py')):
            print(f"Linking extracted CIFAR-10 from {candidate}")
            os.symlink(os.path.join(candidate, 'cifar-10-batches-py'), extracted_target)
            found = True
            break
            
        # Check if tar.gz exists
        tar_path = os.path.join(candidate, 'cifar-10-python.tar.gz')
        if os.path.isfile(tar_path):
            print(f"Copying and extracting {tar_path}")
            shutil.copy(tar_path, tar_target)
            with tarfile.open(tar_target, 'r:gz') as tar:
                tar.extractall(path=target_dir)
            found = True
            break
        elif os.path.isfile(candidate) and candidate.endswith('.tar.gz'):
            print(f"Copying and extracting {candidate}")
            shutil.copy(candidate, tar_target)
            with tarfile.open(tar_target, 'r:gz') as tar:
                tar.extractall(path=target_dir)
            found = True
            break

    if not found:
        print("Warning: Could not find local CIFAR-10 dataset candidate. Will download from torchvision if internet is enabled.")

# Prepare the dataset
prepare_cifar10()

transform = transforms.Compose([transforms.ToTensor()])
print("Loading CIFAR-10 data...")
train_set = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
]))
test_set = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=128, shuffle=False, num_workers=2)



Linking extracted CIFAR-10 from /kaggle/input/datasets/truongnhatquangk18dn/aadata/cifar-10-python
Loading CIFAR-10 data...


In [6]:
# 5. Load Pre-trained Standard Model via RobustBench
try:
    import robustbench
except ImportError:
    print("Installing robustbench from GitHub...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "git+https://github.com/RobustBench/robustbench.git", "--quiet"])

from robustbench.utils import load_model

print("Loading standard ResNet-18 model from RobustBench...")
base_model = load_model(
    model_name='Standard',
    dataset='cifar10',
    threat_model='Linf'
).to(device)
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs with DataParallel!")
    base_model = nn.DataParallel(base_model)
base_model.eval()
print("Model loaded successfully!")




Installing robustbench from GitHub...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 53.0 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

Loading standard ResNet-18 model from RobustBench...


Downloading...
From (original): https://drive.google.com/uc?id=1t98aEuzeTL8P7Kpd5DIrCoCL21BNZUhC
From (redirected): https://drive.google.com/uc?id=1t98aEuzeTL8P7Kpd5DIrCoCL21BNZUhC&confirm=t&uuid=38b0fe95-b653-45c7-8a76-852150bbe5a0
To: /kaggle/working/models/cifar10/Linf/Standard.pt
100%|██████████| 292M/292M [00:05<00:00, 58.0MB/s] 


Using 2 GPUs with DataParallel!
Model loaded successfully!


In [7]:
# 6. Gather Test Dataset (512 samples)
eval_size = 512
eval_images = []
eval_labels = []
samples_collected = 0
for img, lbl in test_loader:
    if samples_collected >= eval_size:
        break
    size_to_add = min(eval_size - samples_collected, img.size(0))
    eval_images.append(img[:size_to_add])
    eval_labels.append(lbl[:size_to_add])
    samples_collected += size_to_add

eval_images = torch.cat(eval_images, dim=0).to(device)
eval_labels = torch.cat(eval_labels, dim=0).to(device)

print(f"\nGenerating adversarial evaluation datasets on {eval_size} samples...")
# Generate target images
with torch.no_grad():
    clean_images = eval_images.clone()

def generate_adv_in_batches(model, attack_fn, images, labels, batch_size=128, **kwargs):
    model.eval()
    adv_list = []
    for i in range(0, images.size(0), batch_size):
        img_batch = images[i:i+batch_size]
        lbl_batch = labels[i:i+batch_size]
        adv_batch = attack_fn(model, img_batch, lbl_batch, **kwargs)
        adv_list.append(adv_batch)
    return torch.cat(adv_list, dim=0)

print("Generating PGD-10 images...")
pgd_images = generate_adv_in_batches(base_model, pgd_attack, eval_images, eval_labels, batch_size=128, eps=8/255, alpha=2/255, iters=10)

k_values = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
sparse_images_dict = {}
for k in k_values:
    print(f"Generating Sparse PGD (k={k}) images...")
    sparse_images_dict[k] = generate_adv_in_batches(base_model, topk_pgd_attack, eval_images, eval_labels, batch_size=128, eps=8/255, alpha=2/255, iters=10, k_ratio=k, dynamic=True)




Generating adversarial evaluation datasets on 512 samples...
Generating PGD-10 images...
Generating Sparse PGD (k=0.0) images...
Generating Sparse PGD (k=0.1) images...
Generating Sparse PGD (k=0.2) images...
Generating Sparse PGD (k=0.3) images...
Generating Sparse PGD (k=0.4) images...
Generating Sparse PGD (k=0.5) images...
Generating Sparse PGD (k=0.6) images...
Generating Sparse PGD (k=0.7) images...
Generating Sparse PGD (k=0.8) images...
Generating Sparse PGD (k=0.9) images...
Generating Sparse PGD (k=1.0) images...


In [8]:
# 7. Evaluate Defenses

def evaluate_defense(defense_fn, test_images, test_labels):
    correct = 0
    total = 0
    batch_size = 64
    for i in range(0, test_images.size(0), batch_size):
        imgs = test_images[i:i+batch_size]
        lbls = test_labels[i:i+batch_size]
        
        # Apply defense pre-processing
        with torch.no_grad():
            if isinstance(defense_fn, RandomizedSmoothingModel):
                outputs = defense_fn(imgs)
            else:
                defended_imgs = defense_fn(imgs)
                outputs = base_model(defended_imgs)
            _, predicted = torch.max(outputs.data, 1)
            correct += (predicted == lbls).sum().item()
            total += lbls.size(0)
    return 100 * correct / total

# Instantiate defenses
defenses = {
    "No Defense": lambda x: x,
    "Median Filter (3x3)": MedianSmoothingDefense(kernel_size=3),
    "Bit Reduction (3-bit)": BitReductionsDefense(bits=3),
    "JPEG Compression (Q75)": JPEGCompressionDefense(quality=75),
    "Random Noise (std=0.02)": RandomNoiseDefense(std=0.02),
    "Randomized Smoothing": RandomizedSmoothingModel(base_model, sigma=0.12, N=50),
    "Feature Denoising": None
}

print("\nEvaluating all defenses...")
results = []
for name, defense in defenses.items():
    print(f"Running evaluation for: {name}")
    # Feature Denoising wrapper needs special handling as it registers internal hooks
    if name == "Feature Denoising":
        defense = FeatureDenoisingWrapper(base_model, kernel_size=3)
        correct = 0
        total = 0
        batch_size = 64
        for i in range(0, eval_images.size(0), batch_size):
            # Clean
            with torch.no_grad():
                outputs = defense(clean_images[i:i+batch_size])
                _, predicted = torch.max(outputs, 1)
                correct += (predicted == eval_labels[i:i+batch_size]).sum().item()
                total += len(predicted)
        acc_clean = 100 * correct / total
        
        correct = 0
        for i in range(0, eval_images.size(0), batch_size):
            # PGD
            with torch.no_grad():
                outputs = defense(pgd_images[i:i+batch_size])
                _, predicted = torch.max(outputs, 1)
                correct += (predicted == eval_labels[i:i+batch_size]).sum().item()
        acc_pgd = 100 * correct / total
        
        # Sparse for all k
        acc_sparse = {}
        for k in k_values:
            correct = 0
            for i in range(0, eval_images.size(0), batch_size):
                with torch.no_grad():
                    outputs = defense(sparse_images_dict[k][i:i+batch_size])
                    _, predicted = torch.max(outputs, 1)
                    correct += (predicted == eval_labels[i:i+batch_size]).sum().item()
            acc_sparse[k] = 100 * correct / total
        
        # Clean up hooks
        defense.remove_hooks()
    else:
        acc_clean = evaluate_defense(defense, clean_images, eval_labels)
        acc_pgd = evaluate_defense(defense, pgd_images, eval_labels)
        acc_sparse = {}
        for k in k_values:
            acc_sparse[k] = evaluate_defense(defense, sparse_images_dict[k], eval_labels)
        
    res_dict = {
        "Defense Method": name,
        "Clean Accuracy": f"{acc_clean:.2f}%",
        "PGD-10 Accuracy": f"{acc_pgd:.2f}%"
    }
    for k in k_values:
        res_dict[f"k={k}"] = f"{acc_sparse[k]:.2f}%"
    results.append(res_dict)




Evaluating all defenses...
Running evaluation for: No Defense
Running evaluation for: Median Filter (3x3)
Running evaluation for: Bit Reduction (3-bit)
Running evaluation for: JPEG Compression (Q75)
Running evaluation for: Random Noise (std=0.02)
Running evaluation for: Randomized Smoothing
Running evaluation for: Feature Denoising


In [9]:
# 8. Print Results
df = pd.DataFrame(results)
print("\n" + "="*70)
print(" PREPROCESSING & CERTIFIED DEFENSES COMPARISON REPORT")
print("="*70)
print(df.to_string(index=False))
print("="*70)
print("\nDiscussion:")
print("- Median Filter (3x3) is generally the most effective classical filter against isolated pixel attacks (Sparse PGD).")
print("- Randomized Smoothing provides certified guarantees but heavily affects clean accuracy on standard models.")
print("- Feature Denoising on a standard model is less effective because sparse perturbations spread in intermediate feature layers.")




 PREPROCESSING & CERTIFIED DEFENSES COMPARISON REPORT
         Defense Method Clean Accuracy PGD-10 Accuracy  k=0.0  k=0.1  k=0.2  k=0.3  k=0.4  k=0.5  k=0.6  k=0.7  k=0.8  k=0.9  k=1.0
             No Defense         94.73%           0.00% 94.73%  8.20%  2.15%  0.98%  0.39%  0.39%  0.20%  0.00%  0.00%  0.00%  0.00%
    Median Filter (3x3)         78.91%          36.91% 78.91% 60.94% 52.54% 46.68% 41.99% 41.80% 38.67% 35.35% 36.52% 34.77% 34.38%
  Bit Reduction (3-bit)         84.57%           3.32% 84.57% 41.02% 24.61% 16.41% 11.13%  6.64%  6.45%  5.27%  4.69%  4.49%  3.32%
 JPEG Compression (Q75)         82.23%          43.16% 82.23% 63.28% 55.47% 51.37% 47.85% 46.88% 43.95% 42.97% 40.62% 39.84% 39.84%
Random Noise (std=0.02)         89.84%           0.00% 89.65% 18.95%  6.25%  3.12%  1.37%  0.59%  0.20%  0.20%  0.20%  0.00%  0.20%
   Randomized Smoothing         20.70%          15.23% 19.92% 18.75% 18.16% 18.16% 16.80% 16.99% 16.21% 15.43% 15.43% 15.23% 15.62%
      Feature Denoisi